# 실험 011 — 192×192 크롭·패딩 제거 / EfficientNet-B0 4모델

`baseline_기본.ipynb`의 셀 구성을 기반으로 작성했습니다. **루트에서 저장한 뒤 위에서부터 전체 실행**합니다.

- 분할 seed 42·검증 15%, 학습 seed 42/43/44/45. Cross-validation 없음.
- 42: 정사각형 / 43: 직사각형 허용 / 44: 정사각형+가림 / 45: 정사각형+원근.
- 크롭의 각 변은 96~192픽셀. 정사각형 50%는 96×96이며 면적으로는 25%입니다.
- 실제 사진에서 크롭 → 192×192 확대. 직사각형은 비율 변형 허용. 크기 맞춤용 패딩 없음.
- 검증·추론은 **192×192 중앙 크롭**. 원근 변환에 따른 모서리 채움은 별도로 남습니다.
- 네 모델 모두 ImageNet 사전학습에서 새로 학습합니다. 과거 실험 체크포인트는 재사용하지 않습니다.
- 완료 후 기본 T0.3 / 전체 T0.5 / T0.3 TTA CSV와 각 CSV에 대응하는 ZIP을 생성합니다.
- 각 ZIP의 기본 실행 명령은 `python predict.py --input <test 폴더> --output pred.csv`입니다.
- 기존 #49·#80과 루트 model.py/predict.py는 변경하지 않습니다.

규정: 전체 파라미터 20M 이하, 테스트 500장 Colab CPU 추론 600초 이내, ZIP 200MB 이하. 테스트 데이터는 학습에 사용하지 않습니다.


## 실행 준비

현재 작업 폴더에 `baseline_test5.ipynb`, `train/6e`, `train/it`, `test`, `sample_submission.csv`가 있어야 합니다.
로컬에서는 기존 jihokamp 커널을 선택합니다. 학습 시작 전에 노트북을 저장하세요.

결과는 `실험결과/011_192크롭_패딩제거/seed_42/`에 저장합니다. 기존 완료 가중치가 있으면 덮어쓰지 않고 중단합니다.
추론 ZIP에는 학습 노트북·모델 구조·가중치·추론 코드·실행 안내를 포함합니다. 추론 중 다운로드하지 않습니다.


## 모듈 불러오기 & 시드 고정

재현성을 위해 모든 난수 시드를 고정합니다. 시드가 고정되어 있어야 "내가 바꾼 것 때문에 점수가 오른 것인지, 운이 좋았던 것인지"를 구분할 수 있습니다.

In [ ]:
import csv
import hashlib
import importlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from PIL import Image


def seed_everything(seed):
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)
	torch.backends.cudnn.deterministic = True
	torch.backends.cudnn.benchmark = False
	os.environ['PYTHONHASHSEED'] = str(seed)


SEED = 42
BASE_DIR = Path.cwd().resolve()
NOTEBOOK_PATH = BASE_DIR / 'baseline_test5.ipynb'
assert NOTEBOOK_PATH.is_file(), f'루트에서 노트북을 저장한 뒤 실행하세요: {NOTEBOOK_PATH}'
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
seed_everything(SEED)


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)
if torch.cuda.is_available():
	print(torch.cuda.get_device_name())


## 하이퍼 파라미터

In [ ]:
params = {
	'epoch': 50,
	'early_stopping': True,
	'min_epochs': 5,
	'patience': 5,
	'batch_size': 32,
	'learning_rates': (0.001, 0.0003, 0.0001),
	'valid_ratio': 0.15,
	'workers': 0,
	'num_show_img': 5,
}


In [ ]:
class_names = {0: '6e', 1: 'it'}
RUN_SEEDS = (42, 43, 44, 45)
AUGMENTATION_BY_SEED = {42: 'square', 43: 'rectangle', 44: 'square_erasing', 45: 'square_perspective'}
RESULT_DIR = BASE_DIR / '실험결과' / '011_192크롭_패딩제거' / 'seed_42'
CHECKPOINT_DIR = RESULT_DIR / 'checkpoints'
EPOCH_DIR = RESULT_DIR / 'epochs'
GRAPH_DIR = RESULT_DIR / 'graphs'
CODE_DIR = RESULT_DIR / '코드'
assert not (RESULT_DIR / 'runs.csv').exists(), f'기존 학습 결과를 덮어쓰지 않습니다: {RESULT_DIR}'
assert not (RESULT_DIR / 'ensemble.pt').exists(), f'기존 가중치를 덮어쓰지 않습니다: {RESULT_DIR}'
assert not list(CHECKPOINT_DIR.glob('*.pt')), f'체크포인트가 이미 있습니다: {CHECKPOINT_DIR}'
for directory in (CHECKPOINT_DIR, EPOCH_DIR, GRAPH_DIR, CODE_DIR):
	directory.mkdir(parents=True, exist_ok=True)
print('결과 위치:', RESULT_DIR)


## 모델·크롭 정의 (`model_test5.py` 생성)

다른 실험과 파일이 섞이지 않도록 별도 모듈을 생성합니다. 제출 ZIP에서는 model.py라는 이름으로 복사합니다.


In [ ]:
%%writefile model_test5.py
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.transforms import functional as F

IMG_SIZE = 192
MIN_CROP_SIZE = 96
NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD = (0.229, 0.224, 0.225)
PERSPECTIVE_FILL = tuple(round(value * 255) for value in NORM_MEAN)
TEMPERATURE_RULES = ("T0.3", "T0.5")


class RandomPhotoCrop:
	def __init__(self, square):
		self.square = square

	def __call__(self, image):
		width, height = image.size
		assert (width >= IMG_SIZE) and (height >= IMG_SIZE), f"192보다 작은 원본입니다: {image.size}"
		crop_width = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
		if self.square:
			crop_height = crop_width
		else:
			crop_height = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
			while (crop_width / crop_height < 0.75) or (crop_width / crop_height > 4 / 3):
				crop_width = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
				crop_height = int(torch.randint(MIN_CROP_SIZE, IMG_SIZE + 1, ()).item())
		top = int(torch.randint(height - crop_height + 1, ()).item())
		left = int(torch.randint(width - crop_width + 1, ()).item())
		return F.crop(image, top, left, crop_height, crop_width)


def center_crop(image):
	width, height = image.size
	assert (width >= IMG_SIZE) and (height >= IMG_SIZE), f"192より小さいためパディングなしで切り出せません: {image.size}"
	return F.center_crop(image, [IMG_SIZE, IMG_SIZE])


def build_valid_transform():
	return transforms.Compose([
		transforms.Lambda(center_crop),
		transforms.ToTensor(),
		transforms.Normalize(NORM_MEAN, NORM_STD),
	])


def combine_probabilities(member_probs, temperature_rule="T0.3"):
	assert (member_probs.ndim == 2) and (member_probs.shape[1] == 4)
	assert bool(torch.isfinite(member_probs).all()), "モデル確率に非有限値があります。"
	assert bool(((member_probs >= 0) & (member_probs <= 1)).all()), "モデル確率が範囲外です。"
	votes = member_probs >= 0.5
	unanimous = (votes == votes[:, :1]).all(dim=1, keepdim=True)
	middle_mean = member_probs.sort(dim=1).values[:, 1:3].mean(dim=1, keepdim=True)
	probs = torch.where(unanimous, member_probs.mean(dim=1, keepdim=True), middle_mean)
	if temperature_rule == "T0.3":
		temperatures = torch.where(unanimous, 0.3, 0.7)
	elif temperature_rule == "T0.5":
		temperatures = 0.5
	else:
		raise ValueError(f"알 수 없는 temperature 규칙: {temperature_rule}")
	return torch.sigmoid(torch.logit(probs) / temperatures)


def build_model(pretrained=False):
	weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
	net = models.efficientnet_b0(weights=weights)
	net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, 1)
	return net


class EfficientNetEnsemble(nn.Module):
	def __init__(self, state_dicts):
		super().__init__()
		assert len(state_dicts) == 4, "앙상블 가중치 4개가 필요합니다."
		self.models = nn.ModuleList()
		for state_dict in state_dicts:
			model = build_model()
			model.load_state_dict(state_dict)
			self.models.append(model)

	def member_probabilities(self, inputs):
		return torch.cat([torch.sigmoid(model(inputs)) for model in self.models], dim=1)

	def forward(self, inputs, temperature_rule="T0.3"):
		return combine_probabilities(self.member_probabilities(inputs), temperature_rule)


In [ ]:
import model_test5 as model_module
importlib.invalidate_caches()
importlib.reload(model_module)
assert Path(model_module.__file__).resolve() == BASE_DIR / 'model_test5.py'
model = model_module.build_model()
print('EfficientNet-B0 / input:', model_module.IMG_SIZE)


### 파라미터 카운터 (20M 제한 확인)

앙상블을 하거나 사전학습 백본을 쓰는 경우 **모든 모델의 파라미터를 합산**한 값이 20,000,000 이하여야 합니다.

In [ ]:
PARAM_LIMIT = 20_000_000
n_params = sum(parameter.numel() for parameter in model.parameters())
print(f'1모델 {n_params:,} / 4모델 {4 * n_params:,} / 제한 {PARAM_LIMIT:,}')
assert 4 * n_params <= PARAM_LIMIT
del model


## 데이터·증강

공통 크롭: 각 변 96~192픽셀, 정사각형 또는 직사각형(가로/세로 0.75~1.333).
직사각형은 192×192로 늘리며 비율을 변형합니다. 크롭 후 좌우 반전·색상 변형·원근을 적용하고 192×192로 맞춥니다.
색상 변형·가림·원근의 강도는 005의 같은 seed 설정을 유지합니다. 가림은 정규화 후 평균값 0으로 채웁니다.


In [ ]:
data_path = BASE_DIR / 'train'
IMG_SIZE = model_module.IMG_SIZE
NORM_MEAN = model_module.NORM_MEAN
NORM_STD = model_module.NORM_STD
AUGMENTATION_SETTINGS = {
	'square': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.0, 'perspective_p': 0.0},
	'rectangle': {'square': False, 'brightness': 0.4, 'contrast': 0.4, 'saturation': 0.3, 'erase_p': 0.0, 'perspective_p': 0.0},
	'square_erasing': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.1, 'perspective_p': 0.0},
	'square_perspective': {'square': True, 'brightness': 0.3, 'contrast': 0.3, 'saturation': 0.2, 'erase_p': 0.0, 'perspective_p': 0.3},
}
train_transforms = {}
for name, settings in AUGMENTATION_SETTINGS.items():
	operations = [
		model_module.RandomPhotoCrop(square=settings['square']),
		transforms.RandomHorizontalFlip(p=0.5),
		transforms.ColorJitter(brightness=settings['brightness'], contrast=settings['contrast'], saturation=settings['saturation']),
	]
	if settings['perspective_p'] > 0:
		operations.append(transforms.RandomPerspective(distortion_scale=0.15, p=settings['perspective_p'], fill=model_module.PERSPECTIVE_FILL))
	operations.extend([
		transforms.Resize((IMG_SIZE, IMG_SIZE)),
		transforms.ToTensor(),
		transforms.Normalize(NORM_MEAN, NORM_STD),
	])
	if settings['erase_p'] > 0:
		operations.append(transforms.RandomErasing(p=settings['erase_p'], scale=(0.02, 0.1), value=0))
	train_transforms[name] = transforms.Compose(operations)
valid_transform = model_module.build_valid_transform()
full_train = datasets.ImageFolder(data_path, train_transforms['square'])
full_valid = datasets.ImageFolder(data_path, valid_transform)
assert full_train.class_to_idx == {'6e': 0, 'it': 1}
assert len(full_train) == 2906, '학습' + f' 사진 수가 다릅니다: {len(full_train)}'
for path, label in full_train.samples:
	with Image.open(path) as image:
		assert min(image.size) >= IMG_SIZE, f'192보다 작은 이미지: {path}'
print('클래스:', full_train.class_to_idx, '/ 전체:', len(full_train), '/ 입력:', IMG_SIZE)


### Train / Valid 분할

클래스 비율이 유지되도록 층화(stratified) 분할합니다. 검증 점수(log loss)가 리더보드 점수의 예측치 역할을 하므로, **검증 데이터를 학습에 섞지 않도록** 주의하세요.

In [ ]:
labels = [label for _, label in full_train.samples]
train_idx, valid_idx = train_test_split(list(range(len(full_train))), test_size=params['valid_ratio'], stratify=labels, random_state=SEED)
assert (len(train_idx) == 2470) and (len(valid_idx) == 436)
assert not set(train_idx).intersection(valid_idx)
data_sets = {'train': Subset(full_train, train_idx), 'valid': Subset(full_valid, valid_idx)}
with (RESULT_DIR / 'validation_split.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.writer(output)
	writer.writerow(('id', 'label'))
	writer.writerows((Path(full_valid.samples[index][0]).stem, labels[index]) for index in valid_idx)
print(f'train: {len(train_idx)} / valid: {len(valid_idx)} / split seed: {SEED}')


In [ ]:
dataloaders = {
	'train': DataLoader(data_sets['train'], batch_size=params['batch_size'], shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(SEED)),
	'valid': DataLoader(data_sets['valid'], batch_size=params['batch_size'], shuffle=False, num_workers=0),
}
print('배치 수:', len(dataloaders['train']), len(dataloaders['valid']))


### 데이터 확인

In [ ]:
def img_show(tensor, title=None):
	arr = tensor.numpy().transpose((1, 2, 0))
	arr = arr * np.array(NORM_STD) + np.array(NORM_MEAN)  # 정규화 되돌리기 (표시용)
	arr = np.clip(arr, 0, 1)
	plt.figure(figsize=(12, 4))
	plt.imshow(arr)
	plt.axis('off')
	if title is not None:
		plt.title(title)
	plt.show()


inputs, classes = next(iter(dataloaders['train']))
out = torchvision.utils.make_grid(inputs[:params['num_show_img']])
img_show(out, title=[class_names[int(c)] for c in classes[:params['num_show_img']]])


## 네 모델 새 학습

기존 체크포인트를 불러오지 않습니다. 각 모델은 ImageNet 가중치에서 시작해 전체 파라미터를 학습합니다.
학습률 0.001 → 0.0003 → 0.0001, 단계별 최소 5 epoch 이후 5회 연속 검증 손실 미개선 시 전환합니다.
마지막 단계 정체 시 종료하며 최대 50 epoch입니다. 각 모델의 전체 최저 검증 손실을 저장합니다.
배치별 진행 막대 없이 epoch별 한 줄과 CSV를 즉시 기록합니다. 학습 시간은 실제 epoch에 따라 달라집니다.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
	"""optimizer가 주어지면 학습, 아니면 평가. (평균 loss, 정확도)를 반환."""
	training = optimizer is not None
	model.train() if training else model.eval()

	total_loss, total_correct, total_n = 0.0, 0, 0
	with torch.set_grad_enabled(training):
		for inputs, targets in loader:
			inputs = inputs.to(device)
			targets = targets.float().unsqueeze(1).to(device)  # (B,) -> (B,1)

			logits = model(inputs)
			loss = criterion(logits, targets)
			assert bool(torch.isfinite(loss)), "학습/검증 손실이 비유한 값입니다."

			if training:
				optimizer.zero_grad()
				loss.backward()
				optimizer.step()

			probs = torch.sigmoid(logits)
			preds = (probs >= 0.5).float()
			total_loss += loss.item() * inputs.size(0)
			total_correct += (preds == targets).sum().item()
			total_n += inputs.size(0)

	return total_loss / total_n, total_correct / total_n


def train_model(model, dataloaders, criterion, optimizer, num_epochs, epoch_csv, early_stopping, min_epochs, patience, learning_rates):
	assert min_epochs > 0
	assert patience > 0
	assert len(learning_rates) > 0
	assert all(rate > 0 for rate in learning_rates)
	since = time.time()
	history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}
	best_acc, best_loss, best_epoch = -float('inf'), float('inf'), -1
	stage_index, stage_epochs = 0, 0
	stage_best_loss = float('inf')
	epochs_without_improvement = 0
	epoch_writer = csv.writer(epoch_csv)
	epoch_writer.writerow(('epoch', 'learning_rate', 'lr_stage', 'stage_epoch', 'train_loss', 'train_acc', 'valid_loss', 'valid_acc'))

	for epoch in range(1, num_epochs + 1):
		learning_rate = learning_rates[stage_index]
		for group in optimizer.param_groups:
			group['lr'] = learning_rate
		stage_epochs += 1
		tr_loss, tr_acc = run_epoch(model, dataloaders['train'], criterion, optimizer)
		va_loss, va_acc = run_epoch(model, dataloaders['valid'], criterion)

		history['train_loss'].append(tr_loss)
		history['train_acc'].append(tr_acc)
		history['valid_loss'].append(va_loss)
		history['valid_acc'].append(va_acc)
		epoch_writer.writerow((epoch, learning_rate, stage_index + 1, stage_epochs, tr_loss, tr_acc, va_loss, va_acc))
		epoch_csv.flush()
		print(f'Epoch {epoch}/{num_epochs} | lr {learning_rate:g} | train loss {tr_loss:.4f}, acc {tr_acc:.3f} | valid loss {va_loss:.4f}, acc {va_acc:.3f}', flush=True)

		if (va_loss < best_loss) or ((va_loss == best_loss) and (va_acc > best_acc)):
			best_acc, best_loss, best_epoch = va_acc, va_loss, epoch
			torch.save(model.state_dict(), pt_file_name)
			print(f'  ↳ best Log Loss 갱신! {pt_file_name} 저장')

		if va_loss < stage_best_loss:
			stage_best_loss = va_loss
			epochs_without_improvement = 0
		elif stage_epochs > min_epochs:
			epochs_without_improvement += 1

		if epochs_without_improvement >= patience:
			if stage_index + 1 < len(learning_rates):
				stage_index += 1
				stage_epochs = 0
				stage_best_loss = float('inf')
				epochs_without_improvement = 0
				print(f'다음 epoch부터 학습률 {learning_rates[stage_index]:g} 적용')
			elif early_stopping:
				print(f'조기 종료: epoch {epoch}, 마지막 학습률에서 Log Loss {patience}회 연속 미개선')
				break

	elapsed = time.time() - since
	print(f'\n학습 완료 ({elapsed // 60:.0f}분 {elapsed % 60:.0f}초)')
	print(f'best: epoch {best_epoch}, valid acc {best_acc:.4f}, log loss {best_loss:.4f}')
	return history, best_epoch, best_loss


In [ ]:
results = []
for filename in ('baseline_test5.ipynb', 'model_test5.py'):
	shutil.copy2(BASE_DIR / filename, CODE_DIR / filename)
source_hashes = {filename: hashlib.sha256((CODE_DIR / filename).read_bytes()).hexdigest() for filename in ('baseline_test5.ipynb', 'model_test5.py')}
training_record = {
	'experiment': '011', 'split_seed': SEED, 'seeds': list(RUN_SEEDS),
	'augmentations': AUGMENTATION_BY_SEED, 'settings': AUGMENTATION_SETTINGS,
	'crop_side_range': [96, 192], 'rectangle_ratio_range': [0.75, 4 / 3],
	'preprocessing': 'center_crop_192_no_padding', 'params': params,
	'python': sys.version, 'torch': torch.__version__, 'torchvision': torchvision.__version__,
	'device': str(device), 'tf32': False, 'source_sha256': source_hashes,
}
(RESULT_DIR / 'training_record.json').write_text(json.dumps(training_record, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
for seed in RUN_SEEDS:
	augmentation = AUGMENTATION_BY_SEED[seed]
	full_run_train = datasets.ImageFolder(data_path, train_transforms[augmentation])
	assert full_run_train.samples == full_train.samples
	run_data_sets = {'train': Subset(full_run_train, train_idx), 'valid': data_sets['valid']}
	print(f'\n학습 시작: seed {seed} / {augmentation}', flush=True)
	seed_everything(seed)
	model = model_module.build_model(pretrained=True).to(device)
	seed_everything(seed)
	dataloaders = {
		'train': DataLoader(run_data_sets['train'], batch_size=params['batch_size'], shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(seed)),
		'valid': DataLoader(run_data_sets['valid'], batch_size=params['batch_size'], shuffle=False, num_workers=0),
	}
	criterion = nn.BCEWithLogitsLoss()
	optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rates'][0])
	pt_file_name = CHECKPOINT_DIR / f'efficientnet_b0_seed{seed}.pt'
	assert not pt_file_name.exists(), f'기존 가중치를 덮어쓰지 않습니다: {pt_file_name}'
	started = time.perf_counter()
	with (EPOCH_DIR / f'efficientnet_b0_seed{seed}.csv').open('w', newline='', encoding='utf-8-sig') as epoch_csv:
		history, best_epoch, best_loss = train_model(model, dataloaders, criterion, optimizer, num_epochs=params['epoch'], epoch_csv=epoch_csv, early_stopping=params['early_stopping'], min_epochs=params['min_epochs'], patience=params['patience'], learning_rates=params['learning_rates'])
	elapsed = time.perf_counter() - started
	results.append({
		'model': 'efficientnet_b0', 'seed': seed, 'augmentation': augmentation,
		'image_size': IMG_SIZE, 'selection_metric': 'valid_loss', 'learning_rates': str(params['learning_rates']),
		'max_epochs': params['epoch'], 'epochs_run': len(history['train_loss']),
		'early_stopping': params['early_stopping'], 'min_epochs': params['min_epochs'], 'patience': params['patience'],
		'best_epoch': best_epoch, 'valid_loss': best_loss, 'valid_acc': history['valid_acc'][best_epoch - 1],
		'seconds': elapsed, 'n_params': n_params, 'checkpoint': str(pt_file_name),
	})
	with (RESULT_DIR / 'runs.csv').open('w', newline='', encoding='utf-8-sig') as output:
		writer = csv.DictWriter(output, fieldnames=list(results[0]))
		writer.writeheader()
		writer.writerows(results)
	epochs = range(1, len(history['train_loss']) + 1)
	fig, axes = plt.subplots(1, 2, figsize=(12, 4))
	for axis, metric in zip(axes, ('loss', 'acc')):
		axis.plot(epochs, history[f'train_{metric}'], label='train')
		axis.plot(epochs, history[f'valid_{metric}'], label='valid')
		axis.set_xlabel('epoch')
		axis.set_ylabel(metric)
		axis.legend()
		axis.grid(alpha=0.3)
	fig.suptitle(f'seed {seed} / {augmentation} / selected epoch {best_epoch}')
	fig.tight_layout()
	fig.savefig(GRAPH_DIR / f'efficientnet_b0_seed{seed}.png', dpi=140)
	plt.close(fig)
	del model, optimizer, dataloaders
	if device.type == 'cuda':
		torch.cuda.empty_cache()
assert len(results) == 4


## 학습 결과·앙상블 검증

4:0 전원 일치 오답과 불일치 오답을 구분해 기록합니다. 서로 다른 증강의 평균·표준편차를 seed 반복 안정성으로 해석하지 않습니다.


In [ ]:
print('학습 완료:', RESULT_DIR / 'runs.csv')
for row in results:
	print(f"seed {row['seed']} / {row['augmentation']} / loss {row['valid_loss']:.6f} / acc {row['valid_acc']:.4%} / {row['seconds']:.1f}s")
checkpoint = {
	'image_size': IMG_SIZE, 'preprocessing': 'center_crop_192_no_padding',
	'seeds': list(RUN_SEEDS), 'augmentations': [AUGMENTATION_BY_SEED[seed] for seed in RUN_SEEDS],
	'state_dicts': [torch.load(CHECKPOINT_DIR / f'efficientnet_b0_seed{seed}.pt', map_location='cpu', weights_only=True) for seed in RUN_SEEDS],
}
torch.save(checkpoint, RESULT_DIR / 'ensemble.pt')
ensemble = model_module.EfficientNetEnsemble(checkpoint['state_dicts']).to(device).eval()
ensemble_params = sum(parameter.numel() for parameter in ensemble.parameters())
assert ensemble_params <= PARAM_LIMIT
valid_loader = DataLoader(data_sets['valid'], batch_size=params['batch_size'], shuffle=False, num_workers=0)
member_batches, tta_batches, target_batches = [], [], []
with torch.inference_mode():
	for inputs, targets in valid_loader:
		inputs = inputs.to(device)
		member_probs = ensemble.member_probabilities(inputs)
		tta_probs = (member_probs + ensemble.member_probabilities(inputs.flip(dims=[3]))) / 2
		member_batches.append(member_probs.cpu())
		tta_batches.append(tta_probs.cpu())
		target_batches.append(targets)
valid_members = torch.cat(member_batches)
valid_tta_members = torch.cat(tta_batches)
valid_targets = torch.cat(target_batches).float()
valid_outputs = {f'seed{seed}': valid_members[:, index] for index, seed in enumerate(RUN_SEEDS)}
valid_outputs['mean_T1'] = valid_members.mean(dim=1)
valid_outputs['T0.3'] = model_module.combine_probabilities(valid_members, 'T0.3').squeeze(1)
valid_outputs['T0.5'] = model_module.combine_probabilities(valid_members, 'T0.5').squeeze(1)
valid_outputs['T0.3_TTA'] = model_module.combine_probabilities(valid_tta_members, 'T0.3').squeeze(1)
metrics = []
for name, probs in valid_outputs.items():
	errors = int(((probs >= 0.5) != valid_targets.bool()).sum())
	loss = torch.nn.functional.binary_cross_entropy(probs.clamp(1e-6, 1 - 1e-6), valid_targets).item()
	metrics.append({'model': name, 'samples': len(probs), 'errors': errors, 'valid_acc': 1 - errors / len(probs), 'valid_loss': loss})
	print(f'{name}: errors {errors} / Log Loss {loss:.8f}')
with (RESULT_DIR / 'ensemble_metrics.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.DictWriter(output, fieldnames=list(metrics[0]))
	writer.writeheader()
	writer.writerows(metrics)
with (RESULT_DIR / 'validation_predictions.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.writer(output)
	writer.writerow(['id', 'label'] + [f'seed{seed}_prob' for seed in RUN_SEEDS] + [f'seed{seed}_tta_prob' for seed in RUN_SEEDS] + ['T0.3', 'T0.5', 'T0.3_TTA'])
	for position, index in enumerate(valid_idx):
		writer.writerow([Path(full_valid.samples[index][0]).stem, int(valid_targets[position])] + valid_members[position].tolist() + valid_tta_members[position].tolist() + [float(valid_outputs[name][position]) for name in ('T0.3', 'T0.5', 'T0.3_TTA')])
agreement_rows = []
for name, members in (('T0.3', valid_members), ('T0.3_TTA', valid_tta_members)):
	votes = (members >= 0.5).sum(dim=1)
	for agreement, mask in (('4:0', (votes == 0) | (votes == 4)), ('3:1', (votes == 1) | (votes == 3)), ('2:2', votes == 2)):
		count = int(mask.sum())
		errors = int(((valid_outputs[name][mask] >= 0.5) != valid_targets[mask].bool()).sum())
		loss = torch.nn.functional.binary_cross_entropy(valid_outputs[name][mask].clamp(1e-6, 1 - 1e-6), valid_targets[mask]).item() if count else None
		agreement_rows.append({'rule': name, 'agreement': agreement, 'samples': count, 'errors': errors, 'valid_loss': loss})
with (RESULT_DIR / 'agreement_metrics.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.DictWriter(output, fieldnames=list(agreement_rows[0]))
	writer.writeheader()
	writer.writerows(agreement_rows)
ensemble = ensemble.cpu()
if device.type == 'cuda':
	torch.cuda.empty_cache()


## 테스트 CSV 생성 — CPU 추론

CPU에서 원본·좌우 반전 확률을 한 번씩 계산하고 세 규칙의 CSV를 생성합니다.
검증·테스트 전처리는 동일한 192×192 중앙 크롭입니다. TTA는 중앙 크롭의 좌우 반전만 추가합니다.


In [ ]:
class TestImageDataset(Dataset):
	def __init__(self, files, transform):
		self.files = files
		self.transform = transform

	def __len__(self):
		return len(self.files)

	def __getitem__(self, index):
		path = self.files[index]
		with Image.open(path) as image:
			return self.transform(image.convert('RGB')), path.stem


test_files = sorted(path for path in (BASE_DIR / 'test').iterdir() if path.is_file() and (path.suffix.lower() in ('.jpg', '.jpeg', '.png')))
with (BASE_DIR / 'sample_submission.csv').open(newline='', encoding='utf-8-sig') as source:
	sample_rows = list(csv.DictReader(source))
expected_ids = {row['id'] for row in sample_rows}
assert (len(test_files) == 500) and (len(expected_ids) == 500)
assert len({path.stem for path in test_files}) == 500
assert {path.stem for path in test_files} == expected_ids
test_loader = DataLoader(TestImageDataset(test_files, valid_transform), batch_size=params['batch_size'], shuffle=False, num_workers=0)


In [ ]:
submission_rows = {name: [] for name in ('T0.3', 'T0.5', 'T0.3_TTA')}
test_member_rows = []
started = time.perf_counter()
with torch.inference_mode():
	for inputs, image_ids in test_loader:
		members = ensemble.member_probabilities(inputs)
		tta_members = (members + ensemble.member_probabilities(inputs.flip(dims=[3]))) / 2
		outputs = {
			'T0.3': model_module.combine_probabilities(members, 'T0.3').squeeze(1),
			'T0.5': model_module.combine_probabilities(members, 'T0.5').squeeze(1),
			'T0.3_TTA': model_module.combine_probabilities(tta_members, 'T0.3').squeeze(1),
		}
		for position, image_id in enumerate(image_ids):
			test_member_rows.append([image_id] + members[position].tolist() + tta_members[position].tolist())
		for name, probs in outputs.items():
			assert bool(torch.isfinite(probs).all())
			submission_rows[name].extend((image_id, f'{prob.item():.6f}') for image_id, prob in zip(image_ids, probs))
test_cpu_seconds = time.perf_counter() - started
for name, rows in submission_rows.items():
	path = RESULT_DIR / f'submission_42_{name}.csv'
	with path.open('w', newline='', encoding='utf-8') as output:
		writer = csv.writer(output)
		writer.writerow(('id', 'prob'))
		writer.writerows(rows)
	assert (len(rows) == 500) and ({row[0] for row in rows} == expected_ids)
	assert all(0 <= float(row[1]) <= 1 for row in rows)
	print('CSV:', path)
with (RESULT_DIR / 'test_member_probabilities.csv').open('w', newline='', encoding='utf-8-sig') as output:
	writer = csv.writer(output)
	writer.writerow(['id'] + [f'seed{seed}_prob' for seed in RUN_SEEDS] + [f'seed{seed}_tta_prob' for seed in RUN_SEEDS])
	writer.writerows(test_member_rows)
print(f'CPU 원본+반전 추론: {test_cpu_seconds:.2f}s / 500장')


## 제출 ZIP 생성

CSV별로 ZIP을 따로 만듭니다. **제출한 CSV와 같은 이름의 ZIP을 선택**하세요.

- submission_42_T0.3.csv ↔ submission_package_T0.3.zip
- submission_42_T0.5.csv ↔ submission_package_T0.5.zip
- submission_42_T0.3_TTA.csv ↔ submission_package_T0.3_TTA.zip

각 ZIP의 `predict.py` 기본값이 해당 CSV 규칙으로 설정됩니다. 운영진은 별도 옵션 없이 같은 명령으로 재현할 수 있습니다.
학습 노트북·최종 가중치·모델 구조·추론 코드·기록·README를 포함하며 모델 생성 시 외부 다운로드가 없습니다.


In [ ]:
%%writefile predict_test5.py
import argparse
import csv
from pathlib import Path

import torch
from PIL import Image

from model_test5 import IMG_SIZE, TEMPERATURE_RULES, EfficientNetEnsemble, build_valid_transform, combine_probabilities

PARAM_LIMIT = 20_000_000


def load_images(input_dir):
	files = sorted(path for path in Path(input_dir).iterdir() if path.is_file() and (path.suffix.lower() in (".jpg", ".jpeg", ".png")))
	assert files, f"이미지가 없습니다: {input_dir}"
	assert len({path.stem for path in files}) == len(files), "이미지 ID가 중복됩니다."
	return files


def main():
	parser = argparse.ArgumentParser()
	parser.add_argument("--input", required=True, help="테스트 이미지 폴더")
	parser.add_argument("--output", required=True, help="출력 CSV 경로")
	parser.add_argument("--weights", default="model.pt", help="앙상블 가중치 파일")
	parser.add_argument("--temperature-rule", choices=TEMPERATURE_RULES, default="T0.3")
	parser.add_argument("--tta", action="store_true", help="모델별 원본·좌우 반전 확률 평균 후 결합")
	args = parser.parse_args()
	weights_path = Path(args.weights)
	if not weights_path.is_absolute():
		weights_path = Path(__file__).resolve().parent / weights_path
	checkpoint = torch.load(weights_path, map_location="cpu", weights_only=True)
	assert checkpoint["image_size"] == IMG_SIZE
	assert checkpoint["preprocessing"] == "center_crop_192_no_padding"
	assert checkpoint["seeds"] == [42, 43, 44, 45]
	model = EfficientNetEnsemble(checkpoint["state_dicts"]).eval()
	n_params = sum(parameter.numel() for parameter in model.parameters())
	assert n_params <= PARAM_LIMIT, f"파라미터 제한 초과: {n_params:,}"
	transform = build_valid_transform()
	files = load_images(args.input)
	rows = []
	with torch.inference_mode():
		for position in range(0, len(files), 32):
			batch_files = files[position:position + 32]
			images = []
			for path in batch_files:
				with Image.open(path) as image:
					images.append(transform(image.convert("RGB")))
			inputs = torch.stack(images)
			member_probs = model.member_probabilities(inputs)
			if args.tta:
				member_probs = (member_probs + model.member_probabilities(inputs.flip(dims=[3]))) / 2
			probs = combine_probabilities(member_probs, args.temperature_rule).squeeze(1)
			assert bool(torch.isfinite(probs).all())
			for path, prob in zip(batch_files, probs):
				rows.append((path.stem, f"{prob.item():.6f}"))
	with Path(args.output).open("w", newline="", encoding="utf-8") as output:
		writer = csv.writer(output)
		writer.writerow(("id", "prob"))
		writer.writerows(rows)
	print(f"{args.output} 저장 완료 / {len(files)}장 / {n_params:,} parameters / TTA={args.tta}")


if __name__ == "__main__":
	main()


In [ ]:
assert hashlib.sha256((BASE_DIR / 'model_test5.py').read_bytes()).hexdigest() == source_hashes['model_test5.py'], '학습 후 모델 코드를 바꾸지 마세요.'
shutil.copy2(BASE_DIR / 'predict_test5.py', CODE_DIR / 'predict_test5.py')
package_paths = {}
submission_hashes = {name: hashlib.sha256((RESULT_DIR / f'submission_42_{name}.csv').read_bytes()).hexdigest() for name in submission_rows}
for name in submission_rows:
	package_dir = RESULT_DIR / f'package_{name}'
	assert not package_dir.exists(), f'기존 패키지를 덮어쓰지 않습니다: {package_dir}'
	package_dir.mkdir()
	shutil.copy2(CODE_DIR / 'model_test5.py', package_dir / 'model.py')
	shutil.copy2(CODE_DIR / 'baseline_test5.ipynb', package_dir / 'baseline_test5.ipynb')
	shutil.copy2(RESULT_DIR / 'ensemble.pt', package_dir / 'model.pt')
	shutil.copy2(RESULT_DIR / 'training_record.json', package_dir / 'training_record.json')
	shutil.copy2(RESULT_DIR / 'validation_split.csv', package_dir / 'validation_split.csv')
	source = (CODE_DIR / 'predict_test5.py').read_text(encoding='utf-8').replace('from model_test5 import ', 'from model import ')
	if name == 'T0.5':
		source = source.replace('default="T0.3"', 'default="T0.5"')
	if name == 'T0.3_TTA':
		source = source.replace('action="store_true"', 'action="store_true", default=True')
	(package_dir / 'predict.py').write_text(source, encoding='utf-8')
	readme = f'''실험 011 / CSV: submission_42_{name}.csv
실행: python predict.py --input <test 폴더> --output pred.csv
기본 규칙: {name}. 추가 옵션 없이 대응 CSV를 재현합니다.
전처리: 원본 중앙 192x192 크롭, 패딩 없음, ImageNet 정규화.
TTA: {'원본·좌우 반전 모델별 확률 평균' if name.endswith('TTA') else '미적용'}.
기본 T0.3: 전원 일치 평균/T=0.3, 불일치 가운데 두 확률 평균/T=0.7.
T0.5: 같은 결합 확률에 전체 T=0.5.
학습: ImageNet 사전학습 EfficientNet-B0 4개를 seed 42~45로 새 학습.
전체 파라미터: {ensemble_params:,}. 추론 중 다운로드 없음.
학습 코드: baseline_test5.ipynb. 학습 재현 시 train/test/sample_submission.csv를 준비하고 루트에서 실행.
CSV SHA-256: {submission_hashes[name]}
필요 모듈: Python 3.10+, torch, torchvision, Pillow. 학습에는 numpy/scikit-learn/matplotlib 추가.
학습 환경: {sys.version.split()[0]}, torch {torch.__version__}, torchvision {torchvision.__version__}.
'''
	(package_dir / 'README.txt').write_text(readme, encoding='utf-8')
	zip_path = Path(shutil.make_archive(str(RESULT_DIR / f'submission_package_{name}'), 'zip', package_dir))
	assert zip_path.stat().st_size <= 200_000_000, 'ZIP 200MB 초과'
	with zipfile.ZipFile(zip_path) as archive:
		assert archive.testzip() is None
	package_paths[name] = package_dir
	print('ZIP:', zip_path, f'({zip_path.stat().st_size:,} bytes)')


## 패키지 자가 검증 — 실제 CPU 재현

세 ZIP을 새 폴더에 풀어 **기본 명령만으로** 각 CSV가 재현되는지 확인합니다.
각 실행 600초 제한과 최대 확률 차이 0.00001 이하를 검사합니다. 로컬 CPU 시간은 실제 Colab CPU 시간과 다를 수 있습니다.
검증 완료 시 package_check.json에 CSV·ZIP 해시와 결과를 저장합니다.


In [ ]:
def read_probs(path):
	with Path(path).open(newline='', encoding='utf-8-sig') as source:
		reader = csv.DictReader(source)
		assert reader.fieldnames == ['id', 'prob']
		rows = list(reader)
	assert len(rows) == 500
	assert len({row['id'] for row in rows}) == 500
	assert {row['id'] for row in rows} == expected_ids
	assert all(np.isfinite(float(row['prob'])) and (0 <= float(row['prob']) <= 1) for row in rows)
	return {row['id']: float(row['prob']) for row in rows}


checks = {}
for name in package_paths:
	zip_path = RESULT_DIR / f'submission_package_{name}.zip'
	verify_dir = RESULT_DIR / f'package_verify_{name}'
	assert not verify_dir.exists(), f'검증 폴더가 이미 있습니다: {verify_dir}'
	with zipfile.ZipFile(zip_path) as archive:
		archive.extractall(verify_dir)
	output_path = RESULT_DIR / f'package_check_42_{name}.csv'
	started = time.perf_counter()
	process = subprocess.run([sys.executable, 'predict.py', '--input', str(BASE_DIR / 'test'), '--output', str(output_path)], cwd=verify_dir, capture_output=True, text=True, encoding='utf-8', errors='strict', timeout=600, env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
	seconds = time.perf_counter() - started
	print(process.stdout)
	assert process.returncode == 0, process.stderr
	assert seconds <= 600, f'CPU 600초 초과: {seconds:.2f}'
	original = read_probs(RESULT_DIR / f'submission_42_{name}.csv')
	reproduced = read_probs(output_path)
	max_diff = max(abs(original[image_id] - reproduced[image_id]) for image_id in original)
	assert max_diff <= 0.00001, f'CSV 재현 차이: {max_diff}'
	checks[name] = {'seconds_local_cpu': seconds, 'max_probability_difference': max_diff, 'csv_sha256': submission_hashes[name], 'zip_sha256': hashlib.sha256(zip_path.read_bytes()).hexdigest()}
	print(f'{name}: CPU {seconds:.2f}s / 최대 차이 {max_diff:.8f} / 통과')
(RESULT_DIR / 'package_check.json').write_text(json.dumps(checks, ensure_ascii=False, indent=2), encoding='utf-8')
print('제출 준비 완료:', RESULT_DIR)


## 제출 파일

1. ensemble_metrics.csv에서 기본 T0.3·T0.5·TTA 결과와 오답을 확인합니다.
2. 선택한 submission_42_*.csv를 사이트에 제출합니다. 기존 #49·#80은 보존합니다.
3. 최종 CSV와 대응하는 submission_package_*.zip을 제출합니다.
4. 로컬 자가 검증 통과와 실제 Colab CPU 600초 확인은 구분합니다.

이 실험은 입력 크기·크롭·패딩 처리를 함께 바꿉니다. 기존 005보다 좋아져도 특정 변경 하나의 효과로 단정하지 않습니다.
